# Replicating CLI Step 3: Fetching Supplementary Data

This notebook replicates the logic in `pystocks/ingest/supplementary.py` that fetches macro risk-free and World Bank data.

## 1. Setup and Contracts

In [ ]:
import sqlite3
import pandas as pd
import requests
from io import StringIO
import wbgapi as wb
from pystocks.supplementary_contract import (
    RISK_FREE_SERIES_BY_ECONOMY,
    WORLD_BANK_INDICATOR_MAP,
    normalize_economy_code,
)
from pystocks.config import SQLITE_DB_PATH

print("Risk-free series map:", RISK_FREE_SERIES_BY_ECONOMY)
print("World Bank indicators:", WORLD_BANK_INDICATOR_MAP)

## 2. Identify Target Economies
Step 3 first identifies which countries (economies) are present in the holdings data to determine which World Bank data to fetch.

In [ ]:
with sqlite3.connect(SQLITE_DB_PATH) as conn:
    holdings_countries = pd.read_sql_query(
        """
        SELECT DISTINCT COALESCE(country_code, country) AS economy_code
        FROM holdings_investor_country
        WHERE COALESCE(country_code, country) IS NOT NULL
        """,
        conn,
    )

economy_codes = [
    code
    for code in (
        normalize_economy_code(value)
        for value in holdings_countries["economy_code"].tolist()
    )
    if code
]

print(f"Found {len(economy_codes)} unique normalized economy codes.")
print("Sample:", economy_codes[:10])

## 3. Fetching Risk-free Rates (from FRED)
Uses `https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}`.

In [ ]:
def fetch_fred_sample(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    frame = pd.read_csv(StringIO(response.text))
    frame['trade_date'] = pd.to_datetime(frame.iloc[:, 0])
    frame['nominal_rate'] = pd.to_numeric(frame.iloc[:, 1], errors='coerce') / 100.0
    return frame.dropna().sort_values('trade_date')

# Sample for USA (DTB3)
usa_rf = fetch_fred_sample(RISK_FREE_SERIES_BY_ECONOMY['USA'])
print("USA Risk-Free (FRED - DTB3) sample:")
display(usa_rf.tail(5))

## 4. Fetching World Bank Indicators (using wbgapi)
Fetches annual data for the requested indicators across identified economies.

In [ ]:
indicator_ids = list(WORLD_BANK_INDICATOR_MAP.keys())

wb_rows = list(wb.data.fetch(
    indicator_ids,
    economy=economy_codes,
    numericTimeKeys=True,
))

wb_df = pd.DataFrame(wb_rows)
wb_df

## 5. Storage Summary (Simulated)
In the CLI, these are stored in `supplementary_risk_free_sources` and `supplementary_world_bank_raw`.

In [ ]:
print("Storage targets:")
print("- supplementary_risk_free_sources (replaces existing)")
print("- supplementary_world_bank_raw (replaces existing)")